# FloodNet U-Net modeling and analysis

This notebook trains and compares two multiclass U-Net models using the official FloodNet train/validation/test splits: a ResNet-34 encoder (primary model) and ResNet-18 (simpler baseline). Both use ImageNet-pretrained encoders, the same combined cross-entropy + soft Dice loss, data augmentation, and tuning budget. It selects learning rate (1e-4 or 3e-4) and stopping epoch using validation mIoU with patience 5 and at most 40 epochs. The held-out test split is evaluated only after those choices are fixed.

Inputs are the half-resolution copies produced by `floodnet_preprocessing.ipynb`; this notebook creates them when absent. Evaluation uses overlapping whole-image windows. It reports per-class IoU, mIoU, percent-flooded estimates, and low/moderate/high impact agreement against the ground truth. Impact thresholds are study parameters below and should be confirmed before interpreting those categories.

In [ ]:
# If needed, install dependencies in the active notebook environment:
# %pip install torch torchvision segmentation-models-pytorch pandas matplotlib pillow

from pathlib import Path
import csv, json, math, random, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from PIL import Image, ImageEnhance
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

SEED = 4002
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)

ROOT = Path.cwd()
if ROOT.name == "Scripts" and ROOT.parent.name == "Project_3": ROOT = ROOT.parent
elif ROOT.name != "Project_3": ROOT = ROOT / "Project_3"
DATA_ROOT = ROOT / "Data" / "FloodNet"
RAW_ROOT = DATA_ROOT / "FloodNet-Supervised_v1.0"
PALETTE_XLSX = DATA_ROOT / "ColorMasks-FloodNetv1.0" / "ColorPalette-Values.xlsx"
HALF_ROOT = DATA_ROOT / "processed_half"
RESULTS_ROOT = ROOT / "Results" / "segmentation"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS = {
    "train": ("train/train-org-img", "train/train-label-img", "ColorMasks-TrainSet"),
    "val": ("val/val-org-img", "val/val-label-img", "ColorMasks-ValSet"),
    "test": ("test/test-org-img", "test/test-label-img", "ColorMasks-TestSet"),
}


## Half-resolution data and class palette

Images are resized bilinearly and masks with nearest-neighbor interpolation. The original RGB label values are translated to integer class IDs using the supplied palette workbook; unrecognized mask colors cause a clear error rather than being silently mapped. The output directory is separate from the source data.

In [ ]:
# FloodNet palette workbook values, read using only Python's standard library.
import zipfile, xml.etree.ElementTree as ET

def read_palette_xlsx(path):
    ns = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    with zipfile.ZipFile(path) as zf:
        strings_root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
        strings = ["".join(t.text or "" for t in si.iter(f"{{{ns['m']}}}t")) for si in strings_root.findall("m:si", ns)]
        sheet = ET.fromstring(zf.read("xl/worksheets/sheet1.xml"))
        rows = []
        for row in sheet.findall(".//m:row", ns):
            vals = []
            for cell in row.findall("m:c", ns):
                v = cell.find("m:v", ns)
                vals.append(strings[int(v.text)] if v is not None and cell.get("t") == "s" else (v.text if v is not None else ""))
            rows.append(vals)
    pairs = {}
    for row in rows:
        if len(row) >= 2:
            name = str(row[0]).strip().lower()
            match = re.fullmatch(r"\s*\(?\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\)?\s*", str(row[1]))
            if match: pairs[name] = tuple(map(int, match.groups()))
    if len(pairs) != 10: raise ValueError(f"Expected 10 FloodNet classes in palette, found {len(pairs)}: {pairs}")
    return pairs

CLASS_COLORS = read_palette_xlsx(PALETTE_XLSX)
CLASS_NAMES = list(CLASS_COLORS)
NUM_CLASSES = len(CLASS_NAMES)
COLOR_TO_ID = {rgb: i for i, rgb in enumerate(CLASS_COLORS.values())}
print(list(enumerate(CLASS_COLORS.items())))


def half_size(size): return (max(1, size[0] // 2), max(1, size[1] // 2))

def get_pairs(split):
    image_rel, label_rel, color_dir = SPLITS[split]
    image_dir = RAW_ROOT / image_rel
    masks = DATA_ROOT / "ColorMasks-FloodNetv1.0" / color_dir
    if not masks.exists(): masks = RAW_ROOT / label_rel
    result = []
    for image_path in sorted(image_dir.glob("*.jpg")):
        candidates = [masks / f"{image_path.stem}_lab.png", masks / f"{image_path.stem}.png"]
        mask_path = next((p for p in candidates if p.exists()), None)
        if mask_path is None: continue
        result.append((image_path.stem, image_path, mask_path))
    return result


def ensure_half_resolution():
    counts = {}
    for split in SPLITS:
        pairs = get_pairs(split); written = 0
        for image_id, image_path, mask_path in pairs:
            out_image = HALF_ROOT / split / "images" / f"{image_id}.jpg"
            out_mask = HALF_ROOT / split / "masks" / f"{image_id}_lab.png"
            if out_image.exists() and out_mask.exists(): continue
            out_image.parent.mkdir(parents=True, exist_ok=True); out_mask.parent.mkdir(parents=True, exist_ok=True)
            with Image.open(image_path) as im:
                im.convert("RGB").resize(half_size(im.size), Image.Resampling.BILINEAR).save(out_image, quality=95)
            with Image.open(mask_path) as im:
                im.resize(half_size(im.size), Image.Resampling.NEAREST).save(out_mask)
            written += 1
        counts[split] = len(pairs)
        print(f"{split}: {len(pairs)} official pairs, {written} resized now")
    return counts

OFFICIAL_COUNTS = ensure_half_resolution()
assert OFFICIAL_COUNTS == {"train": 1445, "val": 450, "test": 448}, f"Unexpected paired counts: {OFFICIAL_COUNTS}"


## Data encoding, patching, and augmentation

Each training example is one random 512 × 512 crop from a training image in each epoch. The image and mask receive matching flips and 90-degree rotations; brightness jitter changes only the image. Images are ImageNet-normalized and masks stay as categorical integer IDs.

In [ ]:
TILE = 512
STRIDE = 384  # overlapping inference windows
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def encode_mask(mask_image):
    rgb = np.asarray(mask_image.convert("RGB"), dtype=np.uint8)
    encoded = np.full(rgb.shape[:2], 255, dtype=np.uint8)
    for class_id, color in enumerate(CLASS_COLORS.values()):
        encoded[np.all(rgb == color, axis=-1)] = class_id
    unknown = np.unique(rgb[encoded == 255].reshape(-1, 3), axis=0) if np.any(encoded == 255) else []
    if len(unknown): raise ValueError(f"Unknown RGB mask values: {unknown[:10]}")
    return encoded.astype(np.int64)


def image_tensor_array(image):
    arr = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    return ((arr - MEAN) / STD).transpose(2, 0, 1).astype(np.float32)

class FloodNetTiles(Dataset):
    def __init__(self, split, augment=False):
        self.split, self.augment = split, augment
        self.items = []
        for image_id, _, _ in get_pairs(split):
            ip = HALF_ROOT / split / "images" / f"{image_id}.jpg"
            mp = HALF_ROOT / split / "masks" / f"{image_id}_lab.png"
            if ip.exists() and mp.exists(): self.items.append((image_id, ip, mp))
        if not self.items: raise RuntimeError(f"No resized pairs found for {split}")
    def __len__(self): return len(self.items)
    def __getitem__(self, index):
        image_id, ip, mp = self.items[index]
        with Image.open(ip) as im: image = im.convert("RGB")
        with Image.open(mp) as mm: mask = mm.copy()
        w, h = image.size
        if w < TILE or h < TILE: raise ValueError(f"{image_id} is smaller than {TILE}: {image.size}")
        x, y = (random.randint(0, w-TILE), random.randint(0, h-TILE)) if self.augment else ((w-TILE)//2, (h-TILE)//2)
        box = (x, y, x+TILE, y+TILE)
        image, mask = image.crop(box), mask.crop(box)
        if self.augment:
            if random.random() < .5: image, mask = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT), mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            if random.random() < .5: image, mask = image.transpose(Image.Transpose.FLIP_TOP_BOTTOM), mask.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
            k = random.randrange(4)
            if k: image, mask = image.rotate(90*k), mask.rotate(90*k)
            image = ImageEnhance.Brightness(image).enhance(random.uniform(.8, 1.2))
        return torch.from_numpy(image_tensor_array(image)), torch.from_numpy(encode_mask(mask)), image_id

train_ds = FloodNetTiles("train", augment=True)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=(DEVICE.type == "cuda"))
print("Training tiles per epoch:", len(train_ds))


## Model, loss, and metrics

Both models use the same SMP U-Net decoder, ImageNet encoder weights, optimization setup, and loss. Soft multiclass Dice loss ignores background class 0; cross-entropy covers all classes. Validation and test metrics are computed from aggregated overlapping-window logits over whole images.

In [ ]:
def make_model(encoder_name):
    return smp.Unet(encoder_name=encoder_name, encoder_weights="imagenet", in_channels=3,
                    classes=NUM_CLASSES, activation=None)


def combined_loss(logits, target):
    ce = nn.functional.cross_entropy(logits, target, ignore_index=255)
    probs = torch.softmax(logits, dim=1)
    valid = (target != 255)
    safe_target = target.masked_fill(~valid, 0)
    onehot = nn.functional.one_hot(safe_target, NUM_CLASSES).permute(0, 3, 1, 2).float()
    valid4 = valid[:, None].float()
    probs, onehot = probs * valid4, onehot * valid4
    dims = (0, 2, 3)
    intersection = (probs * onehot).sum(dims)
    denom = probs.sum(dims) + onehot.sum(dims)
    dice_per_class = (2 * intersection + 1e-6) / (denom + 1e-6)
    dice_loss = 1 - dice_per_class[1:].mean()
    return ce + dice_loss


def confusion_add(confusion, truth, pred):
    valid = (truth >= 0) & (truth < NUM_CLASSES)
    inds = NUM_CLASSES * truth[valid].astype(np.int64) + pred[valid].astype(np.int64)
    confusion += np.bincount(inds, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)
    return confusion


def scores_from_confusion(confusion):
    tp = np.diag(confusion).astype(float)
    union = confusion.sum(0) + confusion.sum(1) - tp
    iou = np.divide(tp, union, out=np.full(NUM_CLASSES, np.nan), where=union > 0)
    return {"miou": float(np.nanmean(iou)), "per_class_iou": iou}


def sliding_positions(length, window=TILE, stride=STRIDE):
    if length <= window: return [0]
    xs = list(range(0, length-window+1, stride))
    if xs[-1] != length-window: xs.append(length-window)
    return xs

@torch.inference_mode()
def predict_whole_image(model, image_path, window=TILE, stride=STRIDE):
    with Image.open(image_path) as im: image = im.convert("RGB")
    w, h = image.size
    pad_w, pad_h = max(0, window-w), max(0, window-h)
    if pad_w or pad_h:
        arr = np.asarray(image)
        mode = "reflect" if arr.shape[0] > 1 and arr.shape[1] > 1 else "edge"
        arr = np.pad(arr, ((0,pad_h),(0,pad_w),(0,0)), mode=mode)
        image = Image.fromarray(arr.astype(np.uint8)); w, h = image.size
    sum_logits = np.zeros((NUM_CLASSES, h, w), dtype=np.float32)
    counts = np.zeros((h, w), dtype=np.float32)
    windows, coords = [], []
    for y in sliding_positions(h, window, stride):
        for x in sliding_positions(w, window, stride):
            crop = image.crop((x,y,x+window,y+window))
            windows.append(image_tensor_array(crop)); coords.append((y,x))
            if len(windows) == 8:
                logits = model(torch.from_numpy(np.stack(windows)).to(DEVICE)).cpu().numpy()
                for logit, (yy,xx) in zip(logits, coords): sum_logits[:,yy:yy+window,xx:xx+window] += logit; counts[yy:yy+window,xx:xx+window] += 1
                windows, coords = [], []
    if windows:
        logits = model(torch.from_numpy(np.stack(windows)).to(DEVICE)).cpu().numpy()
        for logit, (yy,xx) in zip(logits, coords): sum_logits[:,yy:yy+window,xx:xx+window] += logit; counts[yy:yy+window,xx:xx+window] += 1
    return (sum_logits / np.maximum(counts, 1))[ :, :max(h-pad_h,1), :max(w-pad_w,1)].argmax(0)

@torch.inference_mode()
def evaluate_split(model, split):
    model.eval(); confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    records = []
    items = get_pairs(split)
    for idx, (image_id, _, _) in enumerate(items, 1):
        ip = HALF_ROOT / split / "images" / f"{image_id}.jpg"
        mp = HALF_ROOT / split / "masks" / f"{image_id}_lab.png"
        with Image.open(mp) as mm: truth = encode_mask(mm)
        pred = predict_whole_image(model, ip)
        if pred.shape != truth.shape: raise ValueError(f"Size mismatch for {image_id}: {pred.shape} vs {truth.shape}")
        confusion_add(confusion, truth, pred)
        records.append((image_id, pred, truth))
        if idx % 50 == 0: print(f"{split}: {idx}/{len(items)} images", end="\r")
    print(f"{split}: evaluated {len(records)} images")
    return scores_from_confusion(confusion), confusion, records


## Tune both encoder variants using validation only

For each encoder, train at learning rates 1e-4 and 3e-4. Early stopping monitors validation mIoU (patience 5, maximum 40 epochs); the selected learning rate and checkpoint are the configuration with the highest validation mIoU. This grid and stopping rule are identical for both encoders. Save a checkpoint per best configuration.

In [ ]:
LR_GRID = [1e-4, 3e-4]
MAX_EPOCHS = 40
PATIENCE = 5
BATCH_SIZE = 8
val_pairs = get_pairs("val")

@torch.inference_mode()
def validate_miou(model):
    model.eval(); confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for image_id, _, _ in val_pairs:
        with Image.open(HALF_ROOT / "val" / "masks" / f"{image_id}_lab.png") as mm: truth = encode_mask(mm)
        pred = predict_whole_image(model, HALF_ROOT / "val" / "images" / f"{image_id}.jpg")
        confusion_add(confusion, truth, pred)
    return scores_from_confusion(confusion)["miou"]


def fit_one(encoder_name, lr):
    model = make_model(encoder_name).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best_score, best_epoch, stale = -1.0, 0, 0
    checkpoint = RESULTS_ROOT / f"{encoder_name}_lr{lr:.0e}_best.pt"
    history = []
    for epoch in range(1, MAX_EPOCHS+1):
        model.train(); losses = []
        for images, masks, _ in train_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = combined_loss(model(images), masks)
            loss.backward(); optimizer.step(); losses.append(float(loss.detach().cpu()))
        val_miou = validate_miou(model)
        history.append({"encoder": encoder_name, "lr": lr, "epoch": epoch,
                        "train_loss": float(np.mean(losses)), "val_miou": val_miou})
        print(f"{encoder_name} lr={lr:g} epoch={epoch:02d}: loss={np.mean(losses):.4f}, val_mIoU={val_miou:.4f}")
        if val_miou > best_score:
            best_score, best_epoch, stale = val_miou, epoch, 0
            torch.save({"state_dict": model.state_dict(), "encoder": encoder_name,
                        "lr": lr, "epoch": epoch, "val_miou": val_miou}, checkpoint)
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f"Early stopping at epoch {epoch}; best epoch {best_epoch}")
                break
    return {"encoder": encoder_name, "lr": lr, "best_epoch": best_epoch,
            "best_val_miou": best_score, "checkpoint": str(checkpoint)}, history

# This cell runs four training jobs and is compute intensive.
RUN_TUNING = True
if RUN_TUNING:
    tuning_results, tuning_history = [], []
    for encoder in ("resnet34", "resnet18"):
        for lr in LR_GRID:
            result, history = fit_one(encoder, lr)
            tuning_results.append(result); tuning_history.extend(history)
    tuning_table = pd.DataFrame(tuning_results)
    history_df = pd.DataFrame(tuning_history)
    display(tuning_table)
    tuning_table.to_csv(RESULTS_ROOT / "validation_tuning_summary.csv", index=False)
    history_df.to_csv(RESULTS_ROOT / "validation_training_history.csv", index=False)
else:
    print("Set RUN_TUNING=True to train and tune the models.")


## Freeze choices, then evaluate test once

The best validation configuration is selected independently for each encoder. After saving those choices, the following cell performs exactly one test evaluation per chosen model and saves test predictions, aggregate confusion matrices, and per-class metrics. Do not rerun this cell to make modeling choices.

In [ ]:
if not RUN_TUNING: raise RuntimeError("Run tuning before test evaluation")
TEST_LOCK = RESULTS_ROOT / "test_evaluation_done.json"
if TEST_LOCK.exists(): raise RuntimeError(f"Test evaluation already completed and locked: {TEST_LOCK}")
chosen = tuning_table.sort_values("best_val_miou", ascending=False).groupby("encoder", as_index=False).first()
chosen.to_csv(RESULTS_ROOT / "locked_model_choices.csv", index=False)

TEST_RESULTS = {}
for _, row in chosen.iterrows():
    checkpoint = torch.load(row["checkpoint"], map_location=DEVICE, weights_only=False)
    model = make_model(row["encoder"]).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    metrics, confusion, records = evaluate_split(model, "test")
    TEST_RESULTS[row["encoder"]] = {"metrics": metrics, "confusion": confusion, "records": records,
                                    "lr": float(row["lr"]), "best_epoch": int(row["best_epoch"])}
    pred_dir = RESULTS_ROOT / row["encoder"] / "test_predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)
    for image_id, pred, _ in records: np.save(pred_dir / f"{image_id}.npy", pred)
    pd.DataFrame(confusion, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RESULTS_ROOT / f"{row['encoder']}_test_confusion.csv")
    model.to("cpu"); del model

metric_rows = []
for encoder, result in TEST_RESULTS.items():
    metric_rows.append({"encoder": encoder, "test_miou": result["metrics"]["miou"],
                        "learning_rate": result["lr"], "best_epoch": result["best_epoch"]})
    for name, value in zip(CLASS_NAMES, result["metrics"]["per_class_iou"]):
        metric_rows.append({"encoder": encoder, "class": name, "per_class_iou": value})
metric_table = pd.DataFrame(metric_rows)
display(metric_table)
metric_table.to_csv(RESULTS_ROOT / "test_iou_metrics.csv", index=False)
TEST_LOCK.write_text(json.dumps({"completed": True, "models": chosen[["encoder", "lr", "best_epoch", "best_val_miou"]].to_dict(orient="records")}, indent=2))


## Percent flooded and impact-category comparison

We define flooded pixels as `building-flooded`, `road-flooded`, and `water`. Percent flooded is their share among non-background pixels in each image. The ground-truth and predicted values use the same definition. The low/moderate/high cutoffs below are editable analysis assumptions (default: low < 10%, moderate 10–30%, high ≥ 30%); confirm these before interpreting category agreement.

In [ ]:
FLOODED_CLASS_NAMES = {"building-flooded", "road-flooded", "water"}
FLOODED_IDS = np.array([i for i, name in enumerate(CLASS_NAMES) if name in FLOODED_CLASS_NAMES], dtype=int)
BACKGROUND_ID = CLASS_NAMES.index("background")
assert len(FLOODED_IDS) == 3

LOW_MAX_PERCENT = 10.0
MODERATE_MAX_PERCENT = 30.0

def percent_flooded(mask):
    valid = mask != BACKGROUND_ID
    denominator = int(valid.sum())
    return (100.0 * float(np.isin(mask[valid], FLOODED_IDS).sum()) / denominator) if denominator else np.nan

def impact_category(percent):
    if not np.isfinite(percent): return "undefined"
    if percent < LOW_MAX_PERCENT: return "low"
    if percent < MODERATE_MAX_PERCENT: return "moderate"
    return "high"

impact_rows = []
for encoder, result in TEST_RESULTS.items():
    for image_id, prediction, truth in result["records"]:
        pred_pct, true_pct = percent_flooded(prediction), percent_flooded(truth)
        impact_rows.append({"encoder": encoder, "image_id": image_id,
                            "predicted_percent_flooded": pred_pct, "ground_truth_percent_flooded": true_pct,
                            "absolute_percent_error": abs(pred_pct-true_pct) if np.isfinite(pred_pct) and np.isfinite(true_pct) else np.nan,
                            "predicted_impact": impact_category(pred_pct), "ground_truth_impact": impact_category(true_pct)})
impact_df = pd.DataFrame(impact_rows)
impact_df.to_csv(RESULTS_ROOT / "test_flood_impact_by_image.csv", index=False)
impact_summary = (impact_df.groupby("encoder").agg(
    mean_predicted_flood_percent=("predicted_percent_flooded", "mean"),
    mean_ground_truth_flood_percent=("ground_truth_percent_flooded", "mean"),
    mean_absolute_percent_error=("absolute_percent_error", "mean"),
    impact_category_accuracy=("predicted_impact", lambda s: np.mean(s.to_numpy() == impact_df.loc[s.index, "ground_truth_impact"].to_numpy()))
).reset_index())
display(impact_summary)
impact_summary.to_csv(RESULTS_ROOT / "test_flood_impact_summary.csv", index=False)


In [ ]:
# Visual comparisons of the two locked models on a few test images.
example_ids = [r[0] for r in TEST_RESULTS["resnet34"]["records"][:3]]
fig, axes = plt.subplots(len(example_ids), 4, figsize=(16, 4*len(example_ids)), squeeze=False)
CLASS_CMAP = ListedColormap([tuple(np.array(rgb) / 255.0) for rgb in CLASS_COLORS.values()])
for row_i, image_id in enumerate(example_ids):
    with Image.open(HALF_ROOT / "test" / "images" / f"{image_id}.jpg") as im: image = im.convert("RGB")
    truth = next(r[2] for r in TEST_RESULTS["resnet34"]["records"] if r[0] == image_id)
    p34 = next(r[1] for r in TEST_RESULTS["resnet34"]["records"] if r[0] == image_id)
    p18 = next(r[1] for r in TEST_RESULTS["resnet18"]["records"] if r[0] == image_id)
    panels = [(np.asarray(image), "Image"), (truth, "Ground truth"), (p34, "U-Net ResNet-34"), (p18, "Baseline ResNet-18")]
    for col, (arr, title) in enumerate(panels):
        axes[row_i, col].imshow(arr, cmap=None if col == 0 else CLASS_CMAP, vmin=0 if col else None, vmax=NUM_CLASSES-1 if col else None)
        axes[row_i, col].set_title(f"{title} · {image_id}"); axes[row_i, col].axis("off")
plt.tight_layout(); plt.show()


## Analysis record

No significance test or cross-validation is performed. The official split is preserved, the validation set alone determines learning rate/checkpoint selection, and the test split is reserved for final reporting. Training settings, validation history, locked choices, test metrics, confusion matrices, per-image predictions, and flood-impact comparisons are saved under `Project_3/Results/segmentation/`.